# 📊 01 - Data Exploration

**Mục tiêu:** Khám phá dataset video cho bài toán Action Recognition dựa trên skeleton (GCN).

**Nội dung:**
1. Tổng quan dataset (số lượng, class, format)
2. Thống kê video (duration, resolution, FPS)
3. Trực quan hóa phân bố dữ liệu
4. Xem mẫu video từ mỗi class
5. Trích xuất skeleton bằng YOLOv11 Pose
6. Trực quan hóa skeleton trên video mẫu

---
## 1. Setup & Import

In [ ]:
import os
import sys
import glob
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
from collections import Counter, defaultdict
from pathlib import Path
from IPython.display import display, HTML

# Thêm project root vào path
PROJECT_ROOT = Path(r'E:\gcn-action-recognition')
sys.path.insert(0, str(PROJECT_ROOT))

# Đường dẫn data
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

# Matplotlib style
plt.rcParams.update({
    'figure.facecolor': '#1a1a2e',
    'axes.facecolor': '#16213e',
    'axes.edgecolor': '#e94560',
    'axes.labelcolor': '#eee',
    'text.color': '#eee',
    'xtick.color': '#aaa',
    'ytick.color': '#aaa',
    'grid.color': '#333',
    'font.size': 11,
    'figure.dpi': 100,
})

print('✅ Imports thành công!')
print(f'📁 Project root: {PROJECT_ROOT}')
print(f'📁 Data raw: {DATA_RAW}')

---
## 2. Tổng Quan Dataset

In [ ]:
# Quét tất cả video trong data/raw
class_names = sorted([d for d in os.listdir(DATA_RAW) if (DATA_RAW / d).is_dir()])
print(f'🏷️  Số classes: {len(class_names)}')
print(f'📋 Danh sách classes: {class_names}')
print()

# Đếm số video mỗi class
class_video_info = {}
VIDEO_EXTENSIONS = {'.mp4', '.avi', '.mov', '.mkv', '.wmv'}

for cls in class_names:
    cls_path = DATA_RAW / cls
    videos = [f for f in os.listdir(cls_path) 
              if Path(f).suffix.lower() in VIDEO_EXTENSIONS]
    formats = Counter(Path(f).suffix.lower() for f in videos)
    class_video_info[cls] = {
        'count': len(videos),
        'formats': dict(formats),
        'files': sorted(videos, key=lambda x: int(Path(x).stem) if Path(x).stem.isdigit() else x)
    }
    print(f'  📂 {cls:30s} → {len(videos):4d} videos  |  formats: {dict(formats)}')

total_videos = sum(info['count'] for info in class_video_info.values())
print(f'\n📊 Tổng cộng: {total_videos} videos')

In [ ]:
# Biểu đồ số lượng video mỗi class
fig, ax = plt.subplots(figsize=(12, 5))

colors = ['#e94560', '#0f3460', '#533483', '#16c79a', '#f5a623']
counts = [class_video_info[cls]['count'] for cls in class_names]
bars = ax.bar(class_names, counts, color=colors, edgecolor='white', linewidth=0.5, alpha=0.9)

# Thêm số lượng trên mỗi cột
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            str(count), ha='center', va='bottom', fontsize=13, fontweight='bold', color='#eee')

ax.set_title('📊 Số Lượng Video Mỗi Class', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Action Class', fontsize=12)
ax.set_ylabel('Số lượng video', fontsize=12)
ax.set_xticklabels(class_names, rotation=20, ha='right')
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

---
## 3. Thống Kê Chi Tiết Video (Duration, FPS, Resolution)

In [ ]:
def get_video_info(video_path):
    """Trích xuất thông tin cơ bản từ video bằng OpenCV."""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None
    
    info = {
        'width': int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        'height': int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        'fps': cap.get(cv2.CAP_PROP_FPS),
        'total_frames': int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
        'file_size_mb': os.path.getsize(video_path) / (1024 * 1024),
    }
    info['duration_sec'] = info['total_frames'] / info['fps'] if info['fps'] > 0 else 0
    info['resolution'] = f"{info['width']}x{info['height']}"
    
    cap.release()
    return info

print('⏳ Đang thu thập thông tin video... (có thể mất 1-2 phút)')

all_video_stats = []
for cls in class_names:
    cls_path = DATA_RAW / cls
    for filename in class_video_info[cls]['files']:
        video_path = cls_path / filename
        info = get_video_info(video_path)
        if info:
            info['class'] = cls
            info['filename'] = filename
            all_video_stats.append(info)

df_stats = pd.DataFrame(all_video_stats)
print(f'✅ Thu thập xong! Tổng: {len(df_stats)} videos')
print()
df_stats.head()

In [ ]:
# Bảng thống kê tổng hợp theo class
summary = df_stats.groupby('class').agg(
    videos=('filename', 'count'),
    avg_duration=('duration_sec', 'mean'),
    min_duration=('duration_sec', 'min'),
    max_duration=('duration_sec', 'max'),
    avg_fps=('fps', 'mean'),
    avg_frames=('total_frames', 'mean'),
    min_frames=('total_frames', 'min'),
    max_frames=('total_frames', 'max'),
    resolutions=('resolution', lambda x: ', '.join(x.unique())),
    total_size_gb=('file_size_mb', lambda x: x.sum() / 1024),
).round(2)

print('📋 Bảng Thống Kê Tổng Hợp Theo Class:')
print('=' * 120)
display(summary)

In [ ]:
# Thống kê tổng thể
print('📊 THỐNG KÊ TỔNG THỂ DATASET')
print('=' * 50)
print(f'  Tổng số video        : {len(df_stats)}')
print(f'  Tổng thời lượng      : {df_stats["duration_sec"].sum()/60:.1f} phút ({df_stats["duration_sec"].sum()/3600:.2f} giờ)')
print(f'  Duration trung bình  : {df_stats["duration_sec"].mean():.1f}s')
print(f'  Duration min/max     : {df_stats["duration_sec"].min():.1f}s / {df_stats["duration_sec"].max():.1f}s')
print(f'  FPS phổ biến         : {df_stats["fps"].mode().values}')
print(f'  Resolution phổ biến  : {df_stats["resolution"].mode().values}')
print(f'  Tổng frames          : {df_stats["total_frames"].sum():,}')
print(f'  Tổng dung lượng      : {df_stats["file_size_mb"].sum()/1024:.2f} GB')

---
## 4. Trực Quan Hóa Phân Bố Dữ Liệu

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
colors_map = {cls: c for cls, c in zip(class_names, colors)}

# --- 1. Histogram: Duration ---
ax = axes[0, 0]
for cls in class_names:
    data = df_stats[df_stats['class'] == cls]['duration_sec']
    ax.hist(data, bins=20, alpha=0.6, label=cls, color=colors_map[cls], edgecolor='white', linewidth=0.3)
ax.set_title('⏱️ Phân Bố Duration (giây)', fontweight='bold', fontsize=13)
ax.set_xlabel('Duration (s)')
ax.set_ylabel('Số videos')
ax.legend(fontsize=8, loc='upper right')
ax.grid(alpha=0.2)

# --- 2. Histogram: Total Frames ---
ax = axes[0, 1]
for cls in class_names:
    data = df_stats[df_stats['class'] == cls]['total_frames']
    ax.hist(data, bins=20, alpha=0.6, label=cls, color=colors_map[cls], edgecolor='white', linewidth=0.3)
ax.set_title('🎞️ Phân Bố Số Frames', fontweight='bold', fontsize=13)
ax.set_xlabel('Số frames')
ax.set_ylabel('Số videos')
ax.legend(fontsize=8, loc='upper right')
ax.grid(alpha=0.2)

# --- 3. Boxplot: Duration theo class ---
ax = axes[1, 0]
box_data = [df_stats[df_stats['class'] == cls]['duration_sec'].values for cls in class_names]
bp = ax.boxplot(box_data, labels=class_names, patch_artist=True, widths=0.6)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
for element in ['whiskers', 'caps', 'medians']:
    for line in bp[element]:
        line.set_color('#eee')
ax.set_title('📦 Boxplot Duration Theo Class', fontweight='bold', fontsize=13)
ax.set_ylabel('Duration (s)')
ax.set_xticklabels(class_names, rotation=20, ha='right', fontsize=9)
ax.grid(axis='y', alpha=0.2)

# --- 4. Pie chart: Tỷ lệ phân bố class ---
ax = axes[1, 1]
wedges, texts, autotexts = ax.pie(
    counts, labels=class_names, colors=colors, autopct='%1.1f%%',
    startangle=90, pctdistance=0.85,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
for text in texts + autotexts:
    text.set_fontsize(9)
    text.set_color('#eee')
centre_circle = plt.Circle((0, 0), 0.60, fc='#1a1a2e')
ax.add_artist(centre_circle)
ax.set_title('🍩 Tỷ Lệ Phân Bố Class', fontweight='bold', fontsize=13)

fig.suptitle('📊 PHÂN TÍCH PHÂN BỐ DỮ LIỆU', fontsize=18, fontweight='bold', y=1.02, color='#e94560')
plt.tight_layout()
plt.show()

In [ ]:
# Phân bố FPS và Resolution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# FPS distribution
ax = axes[0]
fps_counts = df_stats['fps'].round(0).value_counts().sort_index()
ax.bar(fps_counts.index.astype(str), fps_counts.values, color='#e94560', edgecolor='white', alpha=0.8)
ax.set_title('🎬 Phân Bố FPS', fontweight='bold', fontsize=13)
ax.set_xlabel('FPS')
ax.set_ylabel('Số videos')
for i, (idx, val) in enumerate(fps_counts.items()):
    ax.text(i, val + 1, str(val), ha='center', fontsize=10, color='#eee')
ax.grid(axis='y', alpha=0.2)

# Resolution distribution
ax = axes[1]
res_counts = df_stats['resolution'].value_counts()
ax.bar(res_counts.index, res_counts.values, color='#533483', edgecolor='white', alpha=0.8)
ax.set_title('📐 Phân Bố Resolution', fontweight='bold', fontsize=13)
ax.set_xlabel('Resolution')
ax.set_ylabel('Số videos')
for i, (idx, val) in enumerate(res_counts.items()):
    ax.text(i, val + 1, str(val), ha='center', fontsize=10, color='#eee')
ax.set_xticklabels(res_counts.index, rotation=30, ha='right', fontsize=9)
ax.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.show()

---
## 5. Xem Mẫu Video — Frames Đại Diện Từ Mỗi Class

In [ ]:
def extract_sample_frames(video_path, num_frames=6):
    """Trích xuất num_frames frames cách đều từ video."""
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total <= 0:
        cap.release()
        return []
    
    indices = np.linspace(0, total - 1, num_frames, dtype=int)
    frames = []
    
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame_rgb)
    
    cap.release()
    return frames

# Hiển thị 6 frames đại diện từ 1 video mẫu của mỗi class
fig, axes = plt.subplots(len(class_names), 6, figsize=(20, 4 * len(class_names)))

for row, cls in enumerate(class_names):
    # Chọn video đầu tiên làm mẫu
    sample_file = class_video_info[cls]['files'][0]
    video_path = DATA_RAW / cls / sample_file
    frames = extract_sample_frames(video_path, num_frames=6)
    
    for col in range(6):
        ax = axes[row, col] if len(class_names) > 1 else axes[col]
        if col < len(frames):
            ax.imshow(frames[col])
        ax.axis('off')
        if col == 0:
            ax.set_title(f'🏷️ {cls}', fontsize=12, fontweight='bold', loc='left', color=colors[row % len(colors)])

fig.suptitle('🎬 FRAMES MẪU TỪ MỖI CLASS', fontsize=18, fontweight='bold', y=1.01, color='#e94560')
plt.tight_layout()
plt.show()

---
## 6. Trích Xuất & Trực Quan Skeleton Bằng YOLOv11 Pose

**Model:** `yolo11n-pose` (nano) — 17 COCO keypoints  
**Keypoints:** nose, eyes, ears, shoulders, elbows, wrists, hips, knees, ankles  
**Output format:** `(x_pixel, y_pixel, confidence)` per keypoint

In [ ]:
from ultralytics import YOLO

# Load YOLOv11 Pose Nano
yolo_pose = YOLO('yolo11n-pose.pt')

# 17 COCO keypoints
JOINT_NAMES_17 = [
    'nose',            # 0
    'left_eye',        # 1
    'right_eye',       # 2
    'left_ear',        # 3
    'right_ear',       # 4
    'left_shoulder',   # 5
    'right_shoulder',  # 6
    'left_elbow',      # 7
    'right_elbow',     # 8
    'left_wrist',      # 9
    'right_wrist',     # 10
    'left_hip',        # 11
    'right_hip',       # 12
    'left_knee',       # 13
    'right_knee',      # 14
    'left_ankle',      # 15
    'right_ankle',     # 16
]

# Kết nối giữa các joints (COCO skeleton)
POSE_CONNECTIONS = [
    # Mặt
    (0, 1), (0, 2), (1, 3), (2, 4),
    # Thân trên
    (5, 6),                          # vai
    (5, 7), (7, 9),                  # tay trái
    (6, 8), (8, 10),                 # tay phải
    # Thân dưới
    (5, 11), (6, 12),                # vai → hông
    (11, 12),                        # hông
    (11, 13), (13, 15),              # chân trái
    (12, 14), (14, 16),              # chân phải
]

NUM_JOINTS = 17

print(f'✅ YOLOv11 Pose Nano loaded!')
print(f'📌 Keypoints: {NUM_JOINTS} (COCO format)')
print(f'📌 Output: (x_pixel, y_pixel, confidence) per keypoint')

In [ ]:
def draw_skeleton_on_frame(frame_rgb, keypoints, connections=POSE_CONNECTIONS):
    """Vẽ skeleton lên frame RGB.
    
    Args:
        frame_rgb: frame dạng RGB numpy array (H, W, 3)
        keypoints: np.array shape (17, 3) — [x_pixel, y_pixel, confidence]
        connections: list of (i, j) tuples
    
    Returns:
        annotated: frame RGB với skeleton
    """
    annotated = frame_rgb.copy()
    conf_thresh = 0.5
    
    # Vẽ xương (connections)
    for (i, j) in connections:
        if keypoints[i, 2] > conf_thresh and keypoints[j, 2] > conf_thresh:
            x1, y1 = int(keypoints[i, 0]), int(keypoints[i, 1])
            x2, y2 = int(keypoints[j, 0]), int(keypoints[j, 1])
            cv2.line(annotated, (x1, y1), (x2, y2), (0, 255, 128), 2, cv2.LINE_AA)
    
    # Vẽ joints
    for idx in range(NUM_JOINTS):
        if keypoints[idx, 2] > conf_thresh:
            x, y = int(keypoints[idx, 0]), int(keypoints[idx, 1])
            color = (255, 80, 80) if idx <= 4 else (80, 200, 255)  # đỏ=mặt, xanh=thân
            cv2.circle(annotated, (x, y), 4, color, -1, cv2.LINE_AA)
            cv2.circle(annotated, (x, y), 5, (255, 255, 255), 1, cv2.LINE_AA)
    
    return annotated


def extract_skeleton_from_video_frames(video_path, frame_indices):
    """Trích xuất skeleton từ các frame cụ thể của video bằng YOLO Pose.
    
    Args:
        video_path: đường dẫn video
        frame_indices: list các frame index cần xử lý
    
    Returns:
        results: list of (frame_rgb, keypoints_or_None)
            keypoints: np.array shape (17, 3) — [x_pixel, y_pixel, confidence]
    """
    cap = cv2.VideoCapture(str(video_path))
    results = []
    
    for frame_idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret:
            results.append((None, None))
            continue
        
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # YOLO inference
        det = yolo_pose(frame, verbose=False)
        r = det[0]
        
        keypoints = None
        if r.keypoints is not None and r.keypoints.data.shape[0] > 0:
            # Lấy person có confidence cao nhất (person đầu tiên)
            keypoints = r.keypoints.data[0].cpu().numpy()  # (17, 3)
        
        results.append((frame_rgb, keypoints))
    
    cap.release()
    return results

print('✅ Hàm extract & draw skeleton đã sẵn sàng')

In [ ]:
# Trích xuất skeleton từ 1 video mẫu của mỗi class
NUM_SAMPLE_FRAMES = 6

fig, axes = plt.subplots(len(class_names), NUM_SAMPLE_FRAMES, 
                         figsize=(20, 4 * len(class_names)))

for row, cls in enumerate(class_names):
    sample_file = class_video_info[cls]['files'][0]
    video_path = DATA_RAW / cls / sample_file
    
    # Lấy tổng frames
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    
    indices = np.linspace(0, max(total - 1, 0), NUM_SAMPLE_FRAMES, dtype=int)
    frame_results = extract_skeleton_from_video_frames(video_path, indices)
    
    for col, (frame_rgb, keypoints) in enumerate(frame_results):
        ax = axes[row, col] if len(class_names) > 1 else axes[col]
        
        if frame_rgb is not None:
            if keypoints is not None:
                annotated = draw_skeleton_on_frame(frame_rgb, keypoints)
                ax.imshow(annotated)
                status = '✅'
            else:
                ax.imshow(frame_rgb)
                status = '❌'
            ax.set_xlabel(f'Frame {indices[col]} {status}', fontsize=8, color='#aaa')
        
        ax.axis('off')
        if col == 0:
            ax.set_title(f'🦴 {cls}', fontsize=12, fontweight='bold', 
                       loc='left', color=colors[row % len(colors)])

fig.suptitle('🦴 SKELETON TRÍCH XUẤT TỪ MỖI CLASS (YOLOv11 Pose)', 
             fontsize=18, fontweight='bold', y=1.01, color='#e94560')
plt.tight_layout()
plt.show()

---
## 7. Phân Tích Skeleton Chi Tiết — 1 Video Mẫu

In [ ]:
def extract_all_skeletons(video_path, sample_step=2, max_frames=300):
    """Trích xuất skeleton từ toàn bộ video bằng YOLOv11 Pose.
    
    Args:
        video_path: Đường dẫn video
        sample_step: Lấy mỗi N frame (giảm tải)
        max_frames: Giới hạn số frames tối đa sau khi sample
    
    Returns:
        skeletons: np.array shape (T, 17, 3) — [x_norm, y_norm, confidence]
                   x_norm, y_norm đã được normalize về [0, 1]
        valid_mask: np.array shape (T,) — True nếu detect được skeleton
        video_info: dict thông tin video
    """
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    frame_indices = list(range(0, min(total, max_frames * sample_step), sample_step))
    
    skeletons = []
    valid_mask = []
    
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            break
        
        det = yolo_pose(frame, verbose=False)
        r = det[0]
        
        if r.keypoints is not None and r.keypoints.data.shape[0] > 0:
            kp = r.keypoints.data[0].cpu().numpy()  # (17, 3) — x_pixel, y_pixel, conf
            # Normalize tọa độ về [0, 1]
            kp_norm = kp.copy()
            kp_norm[:, 0] /= w  # x
            kp_norm[:, 1] /= h  # y
            skeletons.append(kp_norm)
            valid_mask.append(True)
        else:
            skeletons.append(np.zeros((NUM_JOINTS, 3)))
            valid_mask.append(False)
    
    cap.release()
    
    return (
        np.array(skeletons),
        np.array(valid_mask),
        {'total_frames': total, 'fps': fps, 'width': w, 'height': h, 
         'sampled_frames': len(skeletons)}
    )

# Test trên 1 video mẫu
test_class = class_names[0]  # first class
test_video = class_video_info[test_class]['files'][0]
test_path = DATA_RAW / test_class / test_video

print(f'🎬 Đang xử lý: {test_class}/{test_video}')
skeletons, valid_mask, vinfo = extract_all_skeletons(test_path, sample_step=3)
print(f'✅ Hoàn thành!')
print(f'   Skeleton shape : {skeletons.shape}  →  (T={skeletons.shape[0]}, V={NUM_JOINTS} joints, C=3 [x,y,conf])')
print(f'   Detect rate    : {valid_mask.sum()}/{len(valid_mask)} ({valid_mask.mean()*100:.1f}%)')
print(f'   Video info     : {vinfo}')

In [ ]:
# Trực quan hóa quỹ đạo joints theo thời gian
# Chọn các keypoints quan trọng
IMPORTANT_JOINTS = {
    0: 'Nose',
    5: 'L.Shoulder',
    6: 'R.Shoulder', 
    7: 'L.Elbow',
    8: 'R.Elbow',
    11: 'L.Hip',
    12: 'R.Hip',
    13: 'L.Knee',
    14: 'R.Knee',
    15: 'L.Ankle',
    16: 'R.Ankle',
}

# Chỉ lấy frames có skeleton
valid_skeletons = skeletons[valid_mask]
T = len(valid_skeletons)
time_axis = np.arange(T)

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# Plot X coordinate
ax = axes[0]
for joint_id, joint_name in IMPORTANT_JOINTS.items():
    ax.plot(time_axis, valid_skeletons[:, joint_id, 0], label=joint_name, alpha=0.7, linewidth=1.2)
ax.set_title(f'📈 Quỹ Đạo X — {test_class}/{test_video}', fontweight='bold', fontsize=13)
ax.set_xlabel('Frame')
ax.set_ylabel('X (normalized 0-1)')
ax.legend(fontsize=7, ncol=4, loc='upper right')
ax.grid(alpha=0.2)

# Plot Y coordinate
ax = axes[1]
for joint_id, joint_name in IMPORTANT_JOINTS.items():
    ax.plot(time_axis, valid_skeletons[:, joint_id, 1], label=joint_name, alpha=0.7, linewidth=1.2)
ax.set_title(f'📈 Quỹ Đạo Y — {test_class}/{test_video}', fontweight='bold', fontsize=13)
ax.set_xlabel('Frame')
ax.set_ylabel('Y (normalized 0-1)')
ax.legend(fontsize=7, ncol=4, loc='upper right')
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: Confidence score trung bình mỗi joint qua toàn bộ video
avg_confidence = valid_skeletons[:, :, 2].mean(axis=0)  # shape (17,)

fig, ax = plt.subplots(figsize=(14, 4))
bar_colors = ['#e94560' if v < 0.5 else '#16c79a' if v > 0.8 else '#f5a623' for v in avg_confidence]
bars = ax.bar(range(NUM_JOINTS), avg_confidence, color=bar_colors, edgecolor='white', linewidth=0.3)
ax.set_xticks(range(NUM_JOINTS))
ax.set_xticklabels(JOINT_NAMES_17, rotation=55, ha='right', fontsize=9)
ax.set_ylabel('Avg Confidence')
ax.set_title(f'👁️ Confidence Score Trung Bình Mỗi Joint — {test_class}/{test_video}', 
             fontweight='bold', fontsize=13)
ax.axhline(y=0.5, color='#e94560', linestyle='--', alpha=0.5, label='Threshold 0.5')
ax.axhline(y=0.8, color='#16c79a', linestyle='--', alpha=0.5, label='Threshold 0.8')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.2)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

---
## 8. So Sánh Skeleton Pattern Giữa Các Class

In [ ]:
# Trích xuất skeleton từ 1 video mỗi class và so sánh
print('⏳ Đang trích xuất skeleton từ mỗi class...')
class_skeletons = {}

for cls in class_names:
    sample_file = class_video_info[cls]['files'][0]
    video_path = DATA_RAW / cls / sample_file
    skel, mask, info = extract_all_skeletons(video_path, sample_step=5, max_frames=100)
    class_skeletons[cls] = {
        'skeletons': skel[mask],  # chỉ lấy valid
        'detect_rate': mask.mean(),
    }
    print(f'  ✅ {cls:30s} → {mask.sum():3d} frames, detect rate: {mask.mean()*100:.1f}%')

print('\n✅ Hoàn thành!')

In [ ]:
# So sánh quỹ đạo Y của các joints chính giữa các class
# → Rất hữu ích để phân biệt standing vs sitting vs falling

compare_joints = {
    0: 'Nose (đầu)',
    11: 'Left Hip (hông)',
    15: 'Left Ankle (mắt cá)',
}

fig, axes = plt.subplots(len(compare_joints), 1, figsize=(16, 4 * len(compare_joints)))

for ax_idx, (joint_id, joint_name) in enumerate(compare_joints.items()):
    ax = axes[ax_idx]
    
    for cls_idx, cls in enumerate(class_names):
        data = class_skeletons[cls]['skeletons']
        if len(data) == 0:
            continue
        
        # Normalize time to 0-1 để so sánh
        T = len(data)
        t_norm = np.linspace(0, 1, T)
        y_values = data[:, joint_id, 1]  # Y coordinate (normalized)
        
        ax.plot(t_norm, y_values, label=cls, color=colors[cls_idx], 
                alpha=0.8, linewidth=1.5)
    
    ax.set_title(f'📊 So Sánh Quỹ Đạo Y: {joint_name}', fontweight='bold', fontsize=13)
    ax.set_xlabel('Time (normalized)')
    ax.set_ylabel('Y (0=top, 1=bottom)')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.2)
    ax.invert_yaxis()  # Y tăng = xuống dưới

fig.suptitle('🔍 SO SÁNH VỊ TRÍ JOINTS GIỮA CÁC ACTION', 
             fontsize=18, fontweight='bold', y=1.02, color='#e94560')
plt.tight_layout()
plt.show()

---
## 9. Tính Motion Energy — Đặc Trưng Chuyển Động

In [ ]:
# Motion energy = tổng biên độ di chuyển của tất cả joints giữa 2 frames liên tiếp
fig, ax = plt.subplots(figsize=(14, 5))

for cls_idx, cls in enumerate(class_names):
    data = class_skeletons[cls]['skeletons']  # (T, 17, 3)
    if len(data) < 2:
        continue
    
    # Tính sự khác biệt giữa 2 frames liên tiếp (chỉ x, y)
    diff = np.diff(data[:, :, :2], axis=0)  # (T-1, 17, 2)
    motion_energy = np.sqrt((diff ** 2).sum(axis=2)).mean(axis=1)  # (T-1,)
    
    t_norm = np.linspace(0, 1, len(motion_energy))
    ax.plot(t_norm, motion_energy, label=f'{cls} (avg={motion_energy.mean():.4f})', 
            color=colors[cls_idx], alpha=0.8, linewidth=1.2)

ax.set_title('⚡ Motion Energy Theo Thời Gian (Mỗi Class 1 Video Mẫu)', 
             fontweight='bold', fontsize=14)
ax.set_xlabel('Time (normalized)')
ax.set_ylabel('Motion Energy (avg joint displacement)')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot so sánh motion energy trung bình
fig, ax = plt.subplots(figsize=(10, 5))

motion_data = []
motion_labels = []

for cls in class_names:
    data = class_skeletons[cls]['skeletons']
    if len(data) < 2:
        continue
    
    diff = np.diff(data[:, :, :2], axis=0)
    motion = np.sqrt((diff ** 2).sum(axis=2)).mean(axis=1)
    motion_data.append(motion)
    motion_labels.append(cls)

bp = ax.boxplot(motion_data, labels=motion_labels, patch_artist=True, widths=0.6)
for patch, color in zip(bp['boxes'], colors[:len(motion_data)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
for element in ['whiskers', 'caps', 'medians']:
    for line in bp[element]:
        line.set_color('#eee')

ax.set_title('📦 Phân Bố Motion Energy Mỗi Class', fontweight='bold', fontsize=14)
ax.set_ylabel('Motion Energy')
ax.set_xticklabels(motion_labels, rotation=20, ha='right')
ax.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.show()

---
## 10. Kết Luận & Nhận Xét Sơ Bộ

In [ ]:
# Tổng hợp kết luận
print('=' * 70)
print('📋 KẾT LUẬN & NHẬN XÉT SƠ BỘ')
print('=' * 70)
print()
print('📊 TỔNG QUAN DATASET:')
print(f'   • Số classes        : {len(class_names)}')
print(f'   • Tổng videos       : {total_videos}')
for cls in class_names:
    print(f'     - {cls:30s}: {class_video_info[cls]["count"]:4d} videos')
print()

# Phân tích cân bằng
max_cls = max(counts)
min_cls = min(counts)
imbalance_ratio = max_cls / min_cls if min_cls > 0 else float('inf')
print(f'⚖️  CÂN BẰNG DỮ LIỆU:')
print(f'   • Class nhiều nhất  : {class_names[counts.index(max_cls)]} ({max_cls} videos)')
print(f'   • Class ít nhất     : {class_names[counts.index(min_cls)]} ({min_cls} videos)')
print(f'   • Tỷ lệ imbalance   : {imbalance_ratio:.2f}x')
if imbalance_ratio > 2:
    print(f'   ⚠️  Dataset khá mất cân bằng, cần xem xét class weights hoặc oversampling')
else:
    print(f'   ✅ Dataset tương đối cân bằng')
print()

# Skeleton quality
print('🦴 CHẤT LƯỢNG SKELETON (YOLOv11 Pose):')
for cls in class_names:
    rate = class_skeletons[cls]['detect_rate']
    status = '✅' if rate > 0.9 else '⚠️' if rate > 0.7 else '❌'
    print(f'   {status} {cls:30s}: detect rate = {rate*100:.1f}%')
print()

print('🔧 THÔNG TIN SKELETON:')
print(f'   • Detector          : YOLOv11n-pose (COCO keypoints)')
print(f'   • Số joints         : {NUM_JOINTS}')
print(f'   • Format            : (x, y, confidence) per keypoint')
print()

print('💡 GỢI Ý TIẾP THEO:')
print('   1. Tiền xử lý: Extract skeleton từ toàn bộ dataset → lưu .npy')
print(f'   2. Data format: (N, C, T, V, M) = (samples, 2, frames, {NUM_JOINTS}_joints, 1_person)')
print('   3. Augmentation: random_choose, random_move, auto_padding từ ST-GCN')
print('   4. Graph: Cần định nghĩa COCO-17 skeleton graph cho ST-GCN')
print('   5. Train/val split: 80/20 stratified')

In [ ]:
# Lưu thống kê ra file CSV để dùng sau
stats_output = PROJECT_ROOT / 'data' / 'video_stats.csv'
df_stats.to_csv(stats_output, index=False)
print(f'💾 Đã lưu thống kê video → {stats_output}')